# Mancino Market: Predictive Regression and Managerial Decisions
## Final integrated notebook | Business Analytics and Data Visualization
Prof. Jose Mendoza · Fall 2026

Which forecasting approach should Mancino use to support daily inventory planning? Compare KNN, regression trees, random forests, neural networks, and averaging ensembles. Select a model using validation evidence, assess its performance on later dates, and translate the results into a managerial recommendation.

Mancino Market is a synthetic grocery retailer with five fictional Manhattan stores. One observation is one store, category, and day. The target is units sold, which can be lower than customer demand when inventory is unavailable.

### How to use this notebook
1. Upload this file through **File > Upload notebook** in [Google Colab](https://colab.research.google.com/). A CPU runtime is sufficient.
2. Run sections 1–5 for data preparation and the baseline.
3. Run sections 6–10 for individual models and ensembles.
4. Section 11 selects the approach using validation MAE. Section 12 refits and tests that selection. Section 13 generates managerial recommendations from the results.

The default notebook runs end to end, including the final test. Before experimenting with settings, set `RUN_FINAL_TEST = False` in section 1. Reopen the test only after choices are complete. Once test results have been inspected, do not use them to retune settings.

This notebook is self-contained and loads the CSV directly. It uses the same historical split as the companion exercises. Reusing that period supports classroom comparisons; it does not supply a new independent test after earlier results have influenced decisions.


## 1. Import the tools
Colab includes these libraries. The notebook prints their versions so results can be traced to the environment used.


In [ ]:
import io
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display, Markdown
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False,
                     'axes.spines.right': False})
print('pandas:', pd.__version__, '| scikit-learn:', sklearn.__version__)

# True runs the complete, predefined workflow through recommendations.
# Set False BEFORE exploring different settings; choose settings on validation data.
RUN_FINAL_TEST = True


## 2. Load and inspect the data
Source: [mancino_daily_sales.csv](https://github.com/jrmst102/mancino/blob/main/data/v1_2025-11-01/mancino_daily_sales.csv).
The verified file contains 22,875 rows, covering January 1–November 1, 2025. The code reports the current file dimensions in case the repository changes.

| Field | Meaning | Use in this notebook |
|---|---|---|
| `date` | Selling date | Split data and derive calendar features |
| `store_id` | Store identifier | Categorical predictor |
| `category` | Product category | Categorical predictor |
| `day_of_week` | Weekday | Recomputed from the date |
| `avg_price` | Observed daily average price | Excluded |
| `units_sold` | Daily units for a store and category | Target |
| `revenue` | Realized sales revenue | Excluded |

Revenue contains information about the sales outcome and would leak information into the model. The observed average price is also excluded because its availability before the selling day has not been established. A verified, planned price could be added in a later exercise.


In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/jrmst102/mancino/main/data/v1_2025-11-01/mancino_daily_sales.csv'
LOCAL_CSV = Path('mancino_daily_sales.csv')

if LOCAL_CSV.exists():
    df = pd.read_csv(LOCAL_CSV)
    print('Loaded local CSV:', LOCAL_CSV)
else:
    try:
        df = pd.read_csv(DATA_URL)
        print('Loaded CSV from GitHub.')
    except Exception as error:
        print('GitHub download failed. Upload mancino_daily_sales.csv when prompted.')
        from google.colab import files
        uploaded = files.upload()
        if 'mancino_daily_sales.csv' not in uploaded:
            raise ValueError('Please upload the named Mancino CSV.') from error
        df = pd.read_csv(io.BytesIO(uploaded['mancino_daily_sales.csv']))

required = {'date', 'store_id', 'category', 'units_sold'}
assert required.issubset(df.columns), 'The CSV is missing required columns.'
df['date'] = pd.to_datetime(df['date'], errors='raise')
assert not df[list(required)].isna().any().any(), 'Check missing required values.'
assert not df.duplicated(['date', 'store_id', 'category']).any(), 'Check duplicate rows.'
assert pd.api.types.is_numeric_dtype(df['units_sold'])
assert df['units_sold'].ge(0).all(), 'Check negative unit sales.'
df = df.sort_values(['date', 'store_id', 'category']).reset_index(drop=True)
print(f'{len(df):,} rows | {df.store_id.nunique()} stores | {df.category.nunique()} categories')
print('Dates:', df.date.min().date(), 'to', df.date.max().date())
display(df.head())
display(df.isna().sum().rename('Missing values').to_frame())


## 3. Define predictors and split by date
Use only information available before the selling day: store, category, weekday, and day of year. Store numbers are identifiers, so they are treated as categories. Day of year supplies a simple numeric time feature; it is not a full seasonality model.

The earliest 60% of dates form the training set, the next 20% form the validation set, and the latest 20% form the test set. Every observation from a given date stays in the same set. This asks whether patterns learned from earlier dates carry forward to later dates for the existing stores and categories.

Train on the training set, choose settings using validation results, and open the test set only after those choices are complete. This notebook does not use lagged sales, promotions, or inventory data.

All rows from the same date stay together. The same training, validation, and test observations are used across model families. Selecting configurations and comparing ensembles on the same validation period can make validation results optimistic; the final test is reserved for assessing the chosen approach.


In [ ]:
df['store_id'] = df['store_id'].astype(str)
df['weekday'] = df['date'].dt.day_name()
df['day_of_year'] = df['date'].dt.dayofyear
CATEGORICAL = ['store_id', 'category', 'weekday']
NUMERIC = ['day_of_year']
FEATURES = CATEGORICAL + NUMERIC
TARGET = 'units_sold'

dates = np.sort(df['date'].unique())
train_end = dates[int(len(dates) * 0.60) - 1]
validation_end = dates[int(len(dates) * 0.80) - 1]
train = df.loc[df.date <= train_end].copy()
validation = df.loc[(df.date > train_end) & (df.date <= validation_end)].copy()
test = df.loc[df.date > validation_end].copy()
assert train.date.max() < validation.date.min() <= validation.date.max() < test.date.min()

X_train, y_train = train[FEATURES], train[TARGET]
X_val, y_val = validation[FEATURES], validation[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
display(pd.DataFrame([
    {'Set': name, 'Rows': len(part), 'First date': part.date.min().date(),
     'Last date': part.date.max().date()}
    for name, part in [('Training', train), ('Validation', validation), ('Test', test)]
]))


In [ ]:
# Explore training outcomes only; keep later outcomes for evaluation.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(y_train, bins=30, color='#356DA8', edgecolor='white')
axes[0].set(xlabel='Daily units per store/category', ylabel='Number of observations',
            title='Training sales distribution')
train.groupby('date').units_sold.sum().plot(ax=axes[1], color='#356DA8')
axes[1].set(xlabel='Date', ylabel='Total units across stores/categories',
            title='Training sales over time')
plt.tight_layout()
plt.show()


## 4. Prepare a consistent comparison
Every candidate uses the same predictors and split. Categories are one-hot encoded. KNN and neural networks standardize day of year; tree-based models keep its original scale. Each pipeline fits its transformations within the training period only.

The neural network also standardizes its target internally, then returns predictions in original unit-sales terms. Ensemble members therefore produce comparable numeric predictions. The baseline is a benchmark, not an ensemble member.

| Metric | Meaning | Preferred direction |
|---|---|---|
| MAE | Average absolute error in units per store/category/day | Lower |
| RMSE | Error in units, giving larger misses more weight | Lower |
| R² | Improvement over predicting the evaluated period's mean | Higher |

R² can be negative and is not classification accuracy. Validation MAE is the selection criterion. Fitting time excludes prediction time and varies with the computing environment. Ensemble fitting time includes refitting all members.


In [ ]:
def make_pipeline(estimator, scale_numeric=False):
    preparation = ColumnTransformer([
        ('categories', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL),
        ('numeric', StandardScaler() if scale_numeric else 'passthrough', NUMERIC)
    ])
    return Pipeline([('prepare', preparation), ('model', estimator)])

def metrics(actual, predicted):
    return {'MAE': mean_absolute_error(actual, predicted),
            'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
            'R2': r2_score(actual, predicted)}

# Rerunning this cell clears all fitted models and earlier evaluation results.
models, results, validation_predictions = {}, {}, {}

def clear_family(family):
    # Discard dependent ensembles whenever a member family is rerun.
    for name in list(results):
        if results[name]['Family'] in {family, 'Ensemble'}:
            for collection in (models, results, validation_predictions):
                collection.pop(name, None)

def fit_and_evaluate(name, family, estimator):
    fitted = clone(estimator)
    started = time.perf_counter()
    fitted.fit(X_train, y_train)
    seconds = time.perf_counter() - started
    train_scores = metrics(y_train, fitted.predict(X_train))
    validation_predictions[name] = fitted.predict(X_val)
    val_scores = metrics(y_val, validation_predictions[name])
    models[name] = fitted
    results[name] = {'Model': name, 'Family': family, 'Train MAE': train_scores['MAE'],
                    'Validation MAE': val_scores['MAE'], 'Validation RMSE': val_scores['RMSE'],
                    'Validation R2': val_scores['R2'], 'Fit seconds': seconds}
    display(pd.DataFrame([results[name]]).round(3))
    return fitted

def best_in_family(family):
    names = [name for name in results if results[name]['Family'] == family]
    assert names, f'Run the {family} section first.'
    return min(names, key=lambda name: results[name]['Validation MAE'])


## 5. Establish a baseline
The baseline always predicts the training set's mean sales. A more complicated model must earn its place by improving on this simple rule. A later extension could use historical averages for each store, category, and weekday as a stronger baseline.


In [ ]:
baseline = fit_and_evaluate('Mean baseline', 'Baseline', DummyRegressor(strategy='mean'))


## 6. KNN regression
KNN predicts by averaging the outcomes of similar training observations. Compare 5, 15, and 50 neighbors with uniform weighting. Smaller neighborhoods follow local variation more closely; larger neighborhoods smooth predictions.

Training predictions can include each row as its own neighbor, so training error is optimistic. Choose *k* using later validation dates. KNN cannot extrapolate beyond the range of its training outcomes.


In [ ]:
K_VALUES = [5, 15, 50]
clear_family('KNN')
for k in K_VALUES:
    assert isinstance(k, int) and 1 <= k <= len(X_train)
    fit_and_evaluate(f'KNN (k={k})', 'KNN', make_pipeline(
        KNeighborsRegressor(n_neighbors=k, weights='uniform', n_jobs=-1), True))


## 7. Regression trees
A regression tree divides observations into groups and predicts the mean sales in each terminal leaf. Compare maximum depths of 3, 6, and 12, keeping the minimum leaf size at 30 observations.

Does greater depth reduce validation error as well as training error? Increasing depth can fit finer patterns, but may also fit random variation. Splits describe predictive associations, not causal effects.


In [ ]:
TREE_DEPTHS = [3, 6, 12]
MIN_LEAF = 30
clear_family('Regression tree')
for depth in TREE_DEPTHS:
    fit_and_evaluate(f'Tree (depth={depth})', 'Regression tree', make_pipeline(
        DecisionTreeRegressor(max_depth=depth, min_samples_leaf=MIN_LEAF, random_state=SEED)))


## 8. Random forests
A random forest averages trees fitted to bootstrap samples, with random subsets of transformed feature columns considered at each split. Compare 50, 100, and 200 trees. Hold maximum depth at 12, minimum leaf size at 30, and `max_features` at 0.8.

More trees generally take longer to fit. Check whether any reduction in validation MAE justifies the additional computation; a larger forest does not guarantee better predictions.


In [ ]:
FOREST_SIZES = [50, 100, 200]
clear_family('Random forest')
for size in FOREST_SIZES:
    fit_and_evaluate(f'Forest (trees={size})', 'Random forest', make_pipeline(
        RandomForestRegressor(n_estimators=size, max_depth=12, min_samples_leaf=30,
                              max_features=0.8, bootstrap=True, random_state=SEED, n_jobs=-1)))


## 9. Neural networks
Compare one hidden layer with 16 neurons, one with 32 neurons, and two hidden layers with 32 and 16 neurons. The networks use ReLU activation, Adam optimization, and a standardized target that is converted back to units for prediction.

Early stopping monitors R² on a random 15% holdout from within the training period. The preprocessing scalers include that internal holdout, so its score is a training diagnostic, not an independent estimate. Chronological validation MAE selects the architecture. Keep convergence warnings visible and address them before opening the test set.


In [ ]:
NETWORK_LAYERS = [(16,), (32,), (32, 16)]
clear_family('Neural network')
for layers in NETWORK_LAYERS:
    estimator = TransformedTargetRegressor(
        regressor=MLPRegressor(hidden_layer_sizes=layers, activation='relu', solver='adam',
                              alpha=0.01, batch_size=256, learning_rate_init=0.001,
                              max_iter=300, early_stopping=True, validation_fraction=0.15,
                              n_iter_no_change=20, random_state=SEED),
        transformer=StandardScaler())
    fitted = fit_and_evaluate(f'Network (hidden={layers})', 'Neural network',
                             make_pipeline(estimator, True))
    print('Training epochs:', fitted.named_steps['model'].regressor_.n_iter_)


## 10. Averaging ensembles
Select the lowest-validation-MAE configuration in each individual model family. Build two ensembles from those four pipelines: an equal-weight average and an illustrative weighted average with relative weights `[1, 1, 2, 2]` for KNN, tree, forest, and network. The baseline is not a member.

Equal weights give each model 25%. The weighted version gives KNN and the tree one-sixth each, and the forest and network one-third each. These weights are specified in advance, not estimated from test results. `VotingRegressor` averages numeric predictions in sales units.

Averaging can help when errors offset, but may dilute a strong model. The residual-correlation table checks whether members tend to make similar errors on validation data. High correlations can limit the benefit of averaging. Selecting members and evaluating ensembles on the same validation period introduces selection optimism; do not treat a tiny validation gain as definitive evidence.


In [ ]:
FAMILIES = ['KNN', 'Regression tree', 'Random forest', 'Neural network']
clear_family('Ensemble')
family_winners = {family: best_in_family(family) for family in FAMILIES}
display(pd.DataFrame([{'Family': family, 'Selected configuration': name,
                      'Validation MAE': results[name]['Validation MAE']}
                     for family, name in family_winners.items()]).round(3))

members = [(f'model_{i}', clone(models[name]))
           for i, name in enumerate(family_winners.values())]
ENSEMBLE_WEIGHTS = [1, 1, 2, 2]
assert len(ENSEMBLE_WEIGHTS) == 4 and np.isfinite(ENSEMBLE_WEIGHTS).all()
assert np.all(np.asarray(ENSEMBLE_WEIGHTS) >= 0) and sum(ENSEMBLE_WEIGHTS) > 0
fit_and_evaluate('Ensemble: equal weights', 'Ensemble',
                 VotingRegressor(estimators=members, n_jobs=1))
fit_and_evaluate('Ensemble: weighted average', 'Ensemble',
                 VotingRegressor(estimators=members, weights=ENSEMBLE_WEIGHTS, n_jobs=1))

residuals = pd.DataFrame({family: y_val.to_numpy() - validation_predictions[name]
                          for family, name in family_winners.items()})
print('Correlations between validation residuals (actual minus predicted):')
display(residuals.corr().round(3))


## 11. Select the model using validation evidence
The leaderboard includes all 12 individual configurations, two ensembles, and the baseline. Select the candidate with the lowest validation MAE. Exact ties retain the first candidate listed in the notebook: the baseline is considered first, then simpler configurations within each family.

The automated selection rule is fixed before test results are inspected. Compare its gain over the baseline, the best individual model, and the best ensemble. A small gain may not justify maintaining four separate pipelines. Fitting times are descriptive and should not be treated as deployment benchmarks.

The final section will distinguish the statistically lowest observed validation MAE from a recommendation to deploy. Record a reason to prefer the selected approach, along with its operational limitation, before the final test.


In [ ]:
assert set(FAMILIES + ['Baseline', 'Ensemble']).issubset({r['Family'] for r in results.values()})
leaderboard = pd.DataFrame(results.values()).sort_values(
    'Validation MAE', kind='stable').reset_index(drop=True)
baseline_val_mae = results['Mean baseline']['Validation MAE']
leaderboard['MAE improvement vs baseline (%)'] = (
    100 * (1 - leaderboard['Validation MAE'] / baseline_val_mae)
    if baseline_val_mae > 0 else np.nan)
display(leaderboard.round(3))

selected_name = min(results, key=lambda name: results[name]['Validation MAE'])
best_individual = min(family_winners.values(), key=lambda name: results[name]['Validation MAE'])
best_ensemble = best_in_family('Ensemble')
selected_family = results[selected_name]['Family']
selected_specification = clone(models[selected_name])
ensemble_gain_units = results[best_individual]['Validation MAE'] - results[best_ensemble]['Validation MAE']
ensemble_gain_percent = 100 * ensemble_gain_units / results[best_individual]['Validation MAE']
print('Model selected before inspecting the test period:', selected_name)
print('Best individual model:', best_individual)
print('Best ensemble:', best_ensemble)
print(f'Ensemble gain over best individual: {ensemble_gain_units:.3f} MAE units ({ensemble_gain_percent:.2f}%).')
print('A negative gain means the ensemble performed worse.')

leaderboard.set_index('Model')[['Train MAE', 'Validation MAE']].plot.barh(
    figsize=(11, 8), color=['#A7C4DE', '#356DA8'])
plt.xlabel('MAE: units per store/category/day')
plt.title('Validation leaderboard across all model families')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 12. Final evaluation on later dates
The selected model specification is now fixed. Refit it on training plus validation data, then evaluate it once on the test period. Refit the mean-sales baseline on that same combined period. No model is selected or retuned using these test outcomes.

For neural-network members, refitting includes a new internal early-stopping holdout within the combined development period. Predictions are returned in original units and are evaluated without rounding or clipping.

The diagnostics report absolute error and signed error. Here, signed error is actual minus predicted sales: a positive mean indicates underprediction. Segment MAE helps identify where errors are largest, but categories with higher volumes may naturally have higher absolute errors. WAPE divides total absolute error by total observed sales to provide context. No metric directly measures profit or a stockout probability.


In [ ]:
test_results = None
if RUN_FINAL_TEST:
    assert 'selected_specification' in globals(), 'Run model selection first.'
    X_development = pd.concat([X_train, X_val])
    y_development = pd.concat([y_train, y_val])
    final_model = clone(selected_specification).fit(X_development, y_development)
    test_predictions = final_model.predict(X_test)
    final_baseline = DummyRegressor(strategy='mean').fit(X_development, y_development)
    baseline_predictions = final_baseline.predict(X_test)
    selected_test_metrics = metrics(y_test, test_predictions)
    baseline_test_metrics = metrics(y_test, baseline_predictions)
    test_results = pd.DataFrame([
        {'Model': selected_name, **selected_test_metrics},
        {'Model': 'Refitted mean baseline', **baseline_test_metrics}
    ])
    print('Validation-selected model:', selected_name)
    display(test_results.round(3))

    diagnostics = test[['date', 'store_id', 'category', TARGET]].copy()
    diagnostics['predicted_units'] = test_predictions
    diagnostics['error'] = y_test.to_numpy() - test_predictions
    diagnostics['absolute_error'] = diagnostics.error.abs()
    diagnostics['underpredicted'] = diagnostics.error > 0
    mean_bias = diagnostics.error.mean()
    underprediction_rate = diagnostics.underpredicted.mean()
    negative_count = int(np.sum(test_predictions < 0))
    test_improvement_pct = (100 * (1 - selected_test_metrics['MAE'] / baseline_test_metrics['MAE'])
                            if baseline_test_metrics['MAE'] > 0 else np.nan)
    display(pd.DataFrame([{
        'Mean signed error (actual - predicted)': mean_bias,
        'Underpredicted observations (%)': 100 * underprediction_rate,
        'Negative predictions': negative_count,
        'MAE improvement over baseline (%)': test_improvement_pct
    }]).round(3))

    def segment_errors(column):
        table = diagnostics.groupby(column).agg(
            Observations=('error', 'size'), Mean_sales=(TARGET, 'mean'),
            MAE=('absolute_error', 'mean'), Mean_signed_error=('error', 'mean'),
            Total_absolute_error=('absolute_error', 'sum'), Total_sales=(TARGET, 'sum'))
        table['WAPE_percent'] = 100 * table.Total_absolute_error / table.Total_sales.replace(0, np.nan)
        return table.drop(columns=['Total_absolute_error', 'Total_sales']).sort_values('MAE', ascending=False)
    store_diagnostics = segment_errors('store_id')
    category_diagnostics = segment_errors('category')
    print('Errors by store:')
    display(store_diagnostics.round(2))
    print('Five categories with the largest absolute errors:')
    display(category_diagnostics.head(5).round(2))
    print('Sample predictions:')
    display(diagnostics[['date', 'store_id', 'category', TARGET, 'predicted_units', 'error']].head(10).round(2))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(y_test, test_predictions, alpha=0.15, s=12, color='#356DA8')
    low, high = min(y_test.min(), test_predictions.min()), max(y_test.max(), test_predictions.max())
    axes[0].plot([low, high], [low, high], color='#B64B44', linestyle='--')
    axes[0].set(xlabel='Actual units', ylabel='Predicted units', title='Final test: predictions')
    daily = diagnostics.groupby('date')[[TARGET, 'predicted_units']].sum()
    daily.plot(ax=axes[1], color=['#356DA8', '#B64B44'])
    axes[1].set(xlabel='Date', ylabel='Units across all stores/categories', title='Daily totals on the test period')
    axes[1].legend(['Actual units', 'Predicted units'])
    plt.tight_layout()
    plt.show()
else:
    print('Final test is disabled. Finish validation choices, then set RUN_FINAL_TEST = True and rerun sections 12–13.')


## 13. Managerial recommendations
The next cell generates an evidence-based briefing using the selected model and test diagnostics. Its operating proposals are recommendations for a controlled pilot, not measured financial benefits.

The model forecasts observed sales, not unconstrained demand or an optimal order quantity. MAE is an average error, not a prediction interval or safety-stock allowance. Inventory on hand, lead times, perishability, service targets, and asymmetric overstock/stockout costs are needed to turn forecasts into replenishment decisions.


In [ ]:
if test_results is None:
    display(Markdown('Managerial recommendations will populate after the final test is enabled and section 12 is run.'))
else:
    worst_store = store_diagnostics.index[0]
    worst_category = category_diagnostics.index[0]
    store_mae = store_diagnostics.iloc[0]['MAE']
    category_mae = category_diagnostics.iloc[0]['MAE']
    category_wape = category_diagnostics.iloc[0]['WAPE_percent']
    bias_word = 'underpredicts' if mean_bias > 0 else 'overpredicts' if mean_bias < 0 else 'has no average signed bias'
    if selected_test_metrics['MAE'] < baseline_test_metrics['MAE']:
        adoption = (f'Use **{selected_name}** as a candidate for a controlled forecasting pilot. '
                    f'It reduced test MAE by **{test_improvement_pct:.1f}%** relative to the refitted mean-sales baseline. '
                    'This is an improvement in forecast error, not a measured reduction in inventory cost.')
    else:
        adoption = (f'Keep the mean-sales baseline as the operational reference. **{selected_name}** was selected on validation data '
                    'but did not improve test MAE. Investigate the generalization gap before piloting a more complex system; '
                    'do not pick a replacement by searching this test period.')
    if selected_family == 'Ensemble':
        complexity = (f'The best ensemble improved validation MAE over **{best_individual}** by '
                      f'**{ensemble_gain_units:.3f} units ({ensemble_gain_percent:.2f}%)**. '
                      'Assess whether that gain justifies maintaining four pipelines. A new future-period evaluation '
                      'is needed before making a data-driven switch to a simpler alternative.')
    else:
        complexity = (f'The validation rule selected **{selected_name}**. Do not add an ensemble solely because it combines more models. '
                      'Record the selected settings and compare any proposed replacement on new future periods.')
    if negative_count:
        output_note = f' The selected model produced {negative_count:,} negative test estimates; define a nonnegative forecasting rule and validate it on new data before operational use.'
    else:
        output_note = ' No negative test estimates were observed.'

    briefing = f"""### Recommended approach
{adoption}

### What the errors mean
Test MAE was **{selected_test_metrics['MAE']:.2f} units per store/category/day**, versus **{baseline_test_metrics['MAE']:.2f}** for the baseline. RMSE was **{selected_test_metrics['RMSE']:.2f} units**, and R² was **{selected_test_metrics['R2']:.3f}**. These are aggregate measures, not guarantees for an individual order.

The model {bias_word} by **{abs(mean_bias):.2f} units on average** across the test observations. Actual sales exceeded predictions in **{100 * underprediction_rate:.1f}%** of observations. This frequency does not equal a stockout rate.{output_note}

### Where managers should review forecasts
Store **{worst_store}** had the largest store-level MAE, **{store_mae:.2f} units**. **{worst_category}** had the largest category-level MAE, **{category_mae:.2f} units**, with WAPE of **{category_wape:.1f}%**. Review those segments first while considering their volumes; higher absolute error alone does not establish worse proportional performance. Use manager review for unusual promotions, disruptions, and inventory constraints.

### Complexity and implementation
{complexity}

### Pilot and data requirements
Run a proposed four-week shadow pilot, producing forecasts without automatically changing orders. Log forecasts before each selling day, compare them with actual sales and a simple historical store/category/weekday baseline, and review MAE and signed bias each week. Measure service levels, stockouts, spoilage, and inventory costs before claiming business value.

Add information available at prediction time: planned promotions and prices, historical sales lags, inventory availability, and supplier lead times. Establish how stockouts censor sales. The current calendar-and-category model is a teaching baseline for existing stores and categories, not a validated demand or replenishment system.

### Student recommendation
In 100–150 words, state the recommended approach, cite its validation and test evidence, identify one priority segment for review, and name one additional data requirement. Distinguish forecast accuracy from financial or operational impact.
"""
    display(Markdown(briefing))


### Data and method references
- [Mancino Market dataset repository](https://github.com/jrmst102/mancino): synthetic teaching data maintained by Dr. Jose Mendoza; CC BY 4.0 according to the repository README.
- [KNeighborsRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html)
- [DecisionTreeRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html)
- [RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)
- [MLPRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html)
- [VotingRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingRegressor.html)

The workflow reports observed error differences without confidence intervals or a significance test. Use rolling evaluations over new time periods before drawing production conclusions.
